<a href="https://colab.research.google.com/github/koushik-ace/NLP/blob/main/Lab11_2_LogisticRegression_Koushik_2403a52258.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1 - Import Libraries

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import re
import nltk
from nltk.corpus import stopwords

# Download NLTK stopwords data if not already present
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


### Step 2 - Load and Preprocess Data

In [2]:
# Create a sample DataFrame (replace with your actual data loading)
data = {
    'text': [
        'This movie was fantastic! I loved every minute of it.',
        'Absolutely terrible. A complete waste of time and money.',
        'It was okay, nothing special but not bad either.',
        'A masterpiece of cinematic art. Highly recommend!',
        'So boring, I almost fell asleep. Skip this one.',
        'Great acting and an engaging plot.',
        'The worst movie I have ever seen. Dreadful!',
        'Highly entertaining and well-directed.',
        'Could have been better, a few good scenes.',
        'Fantastic story, excellent visuals, a must-watch.'
    ],
    'sentiment': [
        'positive', 'negative', 'neutral', 'positive', 'negative',
        'positive', 'negative', 'positive', 'neutral', 'positive'
    ]
}
df = pd.DataFrame(data)


df = df[df['sentiment'] != 'neutral']
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0}) # Convert sentiments to numerical labels

print("Original Data Sample:")
display(df.head())

def preprocess_text(text):
    text = text.lower() # Convert to lowercase
    text = re.sub(r'[^a-z\s]', '', text) # Remove punctuation and numbers
    words = text.split() # Tokenize
    words = [word for word in words if word not in stopwords.words('english')] # Remove stopwords
    return ' '.join(words)

df['processed_text'] = df['text'].apply(preprocess_text)

print("\nSample Processed Text:")
for i in range(3):
    if i < len(df):
        print(f"Original: {df['text'].iloc[i]}")
        print(f"Processed: {df['processed_text'].iloc[i]}\n")



Original Data Sample:


,text,sentiment
0,This movie was fantastic! I loved every minute...,1
1,Absolutely terrible. A complete waste of time ...,0
3,A masterpiece of cinematic art. Highly recommend!,1
4,"So boring, I almost fell asleep. Skip this one.",0
5,Great acting and an engaging plot.,1



Sample Processed Text:
Original: This movie was fantastic! I loved every minute of it.
Processed: movie fantastic loved every minute

Original: Absolutely terrible. A complete waste of time and money.
Processed: absolutely terrible complete waste time money

Original: A masterpiece of cinematic art. Highly recommend!
Processed: masterpiece cinematic art highly recommend



### Step 3 - Feature Extraction


In [3]:
# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limiting to top 5000 features for practical reasons

# Fit and transform the processed text data
X = tfidf_vectorizer.fit_transform(df['processed_text'])
y = df['sentiment'] # Target variable

print(f"Shape of TF-IDF feature matrix: {X.shape}")
print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")



Shape of TF-IDF feature matrix: (8, 36)
Vocabulary size: 36


## Explanation of TF-IDF Weighting
 TF-IDF weighting assigns a score to each word in a document indicating its importance.

 **Term Frequency (TF)**: Measures how frequently a term occurs in a document. The more a word appears, the higher its TF score.

 **Inverse Document Frequency (IDF)**: Measures how unique or rare a term is across the entire corpus. Words that appear in many documents get a lower IDF score (e.g., 'the', 'is'), while rare words get a higher IDF score.


 **TF-IDF Score**: The product of TF and IDF. A high TF-IDF score means the word is frequent in a specific document but rare across other documents, suggesting it's a good descriptor for that document. This helps to filter out common words that don't carry much meaning and emphasize important, distinguishing terms.

### Step 4 - Train Logistic Regression Model
Now that we have our features (X) and target variable (y), we can split our dataset into training and testing sets to evaluate the model's performance on unseen data. Then, we will train a Logistic Regression model, a common and effective algorithm for binary classification tasks.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training data shape: {X_train.shape}, {y_train.shape}")
print(f"Testing data shape: {X_test.shape}, {y_test.shape}")


logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")



Training data shape: (6, 36), (6,)
Testing data shape: (2, 36), (2,)
Logistic Regression model trained successfully.


### Step 5 - Model Evaluation
After training our Logistic Regression model, it's crucial to evaluate its performance using various metrics. This helps us understand how well the model generalizes to unseen data and identify potential areas for improvement. We will calculate accuracy, precision, recall, F1-score, and display a confusion matrix.

In [5]:
# Make predictions on the test set
y_pred = logistic_model.predict(X_test)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Display the metrics
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

# Generate and display the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(conf_matrix)

### Explanation of Evaluation Metrics
# - **Accuracy**: The proportion of correctly classified instances out of the total instances. While intuitive, it can be misleading in imbalanced datasets.
# - **Precision**: The proportion of true positive predictions among all positive predictions. It answers: "Of all instances predicted as positive, how many are actually positive?" High precision means fewer false positives.
# - **Recall (Sensitivity)**: The proportion of true positive predictions among all actual positive instances. It answers: "Of all actual positive instances, how many did we correctly predict as positive?" High recall means fewer false negatives.
# - **F1-Score**: The harmonic mean of precision and recall. It's a balanced metric that considers both false positives and false negatives, especially useful when there's an uneven class distribution.
# - **Confusion Matrix**: A table that visualizes the performance of a classification model. Each row represents the instances in an actual class, while each column represents the instances in a predicted class. It helps differentiate between True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN).

Accuracy: 0.5000
Precision: 0.5000
Recall: 1.0000
F1-Score: 0.6667

Confusion Matrix:
[[0 1]
 [0 1]]


### Step 6 - Train and Evaluate Naive Bayes Model
To compare our Logistic Regression results, we will now train a Naive Bayes classifier, another common algorithm for text classification, especially with TF-IDF features. We will then evaluate its performance using the same metrics.

In [6]:
from sklearn.naive_bayes import MultinomialNB

# Initialize and train the Multinomial Naive Bayes model
naive_bayes_model = MultinomialNB()
naive_bayes_model.fit(X_train, y_train)

print("Multinomial Naive Bayes model trained successfully.")

# Make predictions on the test set
y_pred_nb = naive_bayes_model.predict(X_test)

# Calculate evaluation metrics for Naive Bayes
accuracy_nb = accuracy_score(y_test, y_pred_nb)
precision_nb = precision_score(y_test, y_pred_nb)
recall_nb = recall_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb)

# Display the metrics
print(f"\nNaive Bayes Accuracy: {accuracy_nb:.4f}")
print(f"Naive Bayes Precision: {precision_nb:.4f}")
print(f"Naive Bayes Recall: {recall_nb:.4f}")
print(f"Naive Bayes F1-Score: {f1_nb:.4f}")

# Generate and display the confusion matrix
conf_matrix_nb = confusion_matrix(y_test, y_pred_nb)
print("\nNaive Bayes Confusion Matrix:")
print(conf_matrix_nb)

Multinomial Naive Bayes model trained successfully.

Naive Bayes Accuracy: 0.5000
Naive Bayes Precision: 0.5000
Naive Bayes Recall: 1.0000
Naive Bayes F1-Score: 0.6667

Naive Bayes Confusion Matrix:
[[0 1]
 [0 1]]


### Step 7 - Analysis and Comparison (Logistic Regression vs. Naive Bayes)
Now we can compare the performance of the Logistic Regression model with the Naive Bayes model based on the calculated metrics. This comparison helps us understand which model might be more suitable for this specific dataset and task, or what insights we can gain from their differing strengths and weaknesses.